In [ ]:
!pip install -q transformers datasets seqeval accelerate py_vncorenlp
!pip install -q huggingface_hub
import os
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)
from huggingface_hub import login
import seqeval.metrics
from kaggle_secrets import UserSecretsClient

# Đăng nhập HuggingFace an toàn qua Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN") 
login(token=hf_token)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.3 MB/s eta 0:00:00


In [ ]:
def read_conll_file(file_path):
    sentences = []
    labels = []
    with open(file_path, "r", encoding="utf-8") as f:
        current_words = []
        current_labels = []
        for line in f:
            line = line.strip()
            if not line:
                if current_words:
                    sentences.append(current_words)
                    labels.append(current_labels)
                    current_words = []
                    current_labels = []
            else:
                splits = line.split()
                word = splits[0]
                tag = splits[-1] 
                current_words.append(word)
                current_labels.append(tag)
        
        if current_words:
            sentences.append(current_words)
            labels.append(current_labels)
            
    return sentences, labels

# Đọc các file dữ liệu mà bạn đã upload
train_words, train_tags = read_conll_file('/kaggle/input/datasets/sonbui13/phoner-covid19/word/train_word.conll')
dev_words, dev_tags = read_conll_file('/kaggle/input/datasets/sonbui13/phoner-covid19/word/dev_word.conll')
test_words, test_tags = read_conll_file('/kaggle/input/datasets/sonbui13/phoner-covid19/word/test_word.conll')

# Tạo tập hợp tất cả các nhãn (tags)
unique_tags = set(tag for doc in train_tags for tag in doc)
label_list = sorted(list(unique_tags))
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

print("Danh sách nhãn:", label_list)

Danh sách nhãn: ['B-AGE', 'B-DATE', 'B-GENDER', 'B-JOB', 'B-LOCATION', 'B-NAME', 'B-ORGANIZATION', 'B-PATIENT_ID', 'B-SYMPTOM_AND_DISEASE', 'B-TRANSPORTATION', 'I-AGE', 'I-DATE', 'I-JOB', 'I-LOCATION', 'I-NAME', 'I-ORGANIZATION', 'I-PATIENT_ID', 'I-SYMPTOM_AND_DISEASE', 'I-TRANSPORTATION', 'O']


In [3]:
from transformers import PhobertTokenizerFast
# Chọn model: 'vinai/phobert-base' hoặc 'vinai/phobert-large'
MODEL_CHECKPOINT = "vinai/phobert-large" 

tokenizer = PhobertTokenizerFast.from_pretrained(MODEL_CHECKPOINT)

def convert_to_dataset(words, tags):
    return Dataset.from_dict({"tokens": words, "ner_tags": [[label2id[t] for t in tag] for tag in tags]})

raw_datasets = DatasetDict({
    "train": convert_to_dataset(train_words, train_tags),
    "validation": convert_to_dataset(dev_words, dev_tags),
    "test": convert_to_dataset(test_words, test_tags)
})

def tokenize_and_align_labels(examples):
    all_input_ids = []
    all_attention_mask = []
    all_labels = []

    for i in range(len(examples["tokens"])):
        words = examples["tokens"][i]
        tags = examples["ner_tags"][i]
        
        # Bắt đầu với token mở đầu <s> (ID là 0)
        input_ids = [tokenizer.cls_token_id]
        label_ids = [-100] # Không tính loss cho token <s>

        for word, tag in zip(words, tags):
            # Tokenize từng từ một
            word_tokens = tokenizer.encode(word, add_special_tokens=False)
            
            if len(word_tokens) > 0:
                input_ids.extend(word_tokens)
                # Token đầu tiên của từ nhận nhãn gốc, các sub-tokens sau nhận -100
                label_ids.extend([tag] + [-100] * (len(word_tokens) - 1))

        # Kết thúc với token đóng </s> (ID là 2)
        input_ids.append(tokenizer.sep_token_id)
        label_ids.append(-100)

        # Cắt ngắn (Truncation) nếu vượt quá max_length
        max_length = 256
        if len(input_ids) > max_length:
            input_ids = input_ids[:max_length]
            label_ids = label_ids[:max_length]
            attention_mask = [1] * max_length
        else:
            attention_mask = [1] * len(input_ids)

        all_input_ids.append(input_ids)
        all_attention_mask.append(attention_mask)
        all_labels.append(label_ids)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_mask,
        "labels": all_labels
    }

# CHÚ Ý: Cập nhật lại model checkpoint nếu cần và chạy map
tokenized_datasets = raw_datasets.map(tokenize_and_align_labels, batched=True)

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/5027 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [4]:
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Loại bỏ các vị trí có nhãn -100 (padding/sub-tokens)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # seqeval tính micro-average mặc định khi gọi như thế này
    return {
        "micro_precision": precision_score(true_labels, true_predictions),
        "micro_recall": recall_score(true_labels, true_predictions),
        "micro_f1": f1_score(true_labels, true_predictions),
    }

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)
data_collator = DataCollatorForTokenClassification(tokenizer)

config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.48G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.48G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: vinai/phobert-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:

BATCH_SIZE = 8
GRADIENT_ACCUMULATION = 4  # 8 x 4 = 32
# ==========================================

args = TrainingArguments(
    output_dir="phobert-large-covid19", 
    eval_strategy="epoch",      # Đánh giá sau mỗi epoch
    save_strategy="epoch", # Lưu model sau mỗi epoch
    learning_rate=5e-5,               # LR theo chuẩn paper
    fp16=True,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    num_train_epochs=30,              # Tổng số Epochs là 30
    weight_decay=0.01,
    load_best_model_at_end=True,      # Tải lại model tốt nhất ở cuối
    metric_for_best_model="micro_f1",       # Dùng điểm F1 để chọn model
    push_to_hub=True,                 # Push lên HuggingFace Hub tránh mất dữ liệu
    hub_model_id="ChisDong/phobert_large_IS252", # Sửa lại tên ID HF của bạn

    save_total_limit=2,      # QUAN TRỌNG: Chỉ giữ 2 checkpoint tốt nhất. 
                             # Tránh đầy bộ nhớ 20GB của Kaggle dẫn đến lỗi JSON.
    logging_steps=200,       # Giảm tần suất in log ra màn hình để file ipynb nhẹ hơn
    report_to="none",        # Tắt kết nối bên thứ 3 để tránh treo mạng
    disable_tqdm=True        # Tắt thanh progress bar khi "Save Version" để file log gọn sạch
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    # Áp dụng Early Stopping: dừng nếu f1 không tăng sau 5 epochs liên tiếp
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

In [6]:
# Bắt đầu quá trình huấn luyện
trainer.train()

# Đánh giá trên tập Test
test_results = trainer.evaluate(tokenized_datasets["test"])
print("Kết quả trên tập Test:", test_results)

# Push phiên bản tốt nhất và tokenizer lên Hugging Face Hub (Final push)
trainer.push_to_hub()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': '0.2527', 'eval_micro_precision': '0.8824', 'eval_micro_recall': '0.9207', 'eval_micro_f1': '0.9011', 'eval_runtime': '29.58', 'eval_samples_per_second': '67.62', 'eval_steps_per_second': '4.226', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': '0.1526', 'eval_micro_precision': '0.9254', 'eval_micro_recall': '0.9591', 'eval_micro_f1': '0.9419', 'eval_runtime': '29.49', 'eval_samples_per_second': '67.82', 'eval_steps_per_second': '4.239', 'epoch': '2'}


...

{'eval_loss': '0.2006', 'eval_micro_precision': '0.9496', 'eval_micro_recall': '0.956', 'eval_micro_f1': '0.9528', 'eval_runtime': '44.13', 'eval_samples_per_second': '67.98', 'eval_steps_per_second': '4.26', 'epoch': '15'}
Kết quả trên tập Test: {'eval_loss': 0.20057186484336853, 'eval_micro_precision': 0.9495556495979687, 'eval_micro_recall': 0.956028973157222, 'eval_micro_f1': 0.9527813163481954, 'eval_runtime': 44.1285, 'eval_samples_per_second': 67.983, 'eval_steps_per_second': 4.26, 'epoch': 15.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/ChisDong/phobert_large_IS252/commit/c4faf748daedb790383e3a3e98eeade8244ce4a2', commit_message='End of training', commit_description='', oid='c4faf748daedb790383e3a3e98eeade8244ce4a2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ChisDong/phobert_large_IS252', endpoint='https://huggingface.co', repo_type='model', repo_id='ChisDong/phobert_large_IS252'), pr_revision=None, pr_num=None)